# Minutas verificables con agentes (ReAct + auditor) - Entrega 3

Ejecuta en Google Colab con `Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU (T4)`. Modelo: `ministral-3:3b-instruct-2512-q4_K_M` vía Ollama. Edita solo la celda 1 y luego `Ejecutar todo`. Con `SOURCE = 'git'` (por defecto) el código y el dataset salen del repositorio y no hay que subir nada; con `SOURCE = 'zip'` Colab pide subir archivos en las celdas 3 y 4.

**Modos (`MODE`):**
- `DEMO`: una sola reunión (M01) del split *test*, condición `full`. Es la corrida para el video: la celda 6 imprime la traza agente por agente.
- `VALIDATION`: split *validation*, condiciones `full` y `d2`, limitado a `LIMIT` reuniones (2 por defecto). Prueba de humo para ver si el modelo sigue el protocolo ReAct.
- `FULL`: split *test* completo, las 8 condiciones (`baseline, d2, full, no_react, no_auditor, no_fewshot, no_cot, no_guards`). Es la evaluación final.

**Código:** con `SOURCE = 'zip'` se sube `deliverable3_colab.zip` (generado con `python scripts/package_for_colab.py`); con `SOURCE = 'git'` se clona `REPO_URL` y se fija `COMMIT`. Las respuestas del LLM se guardan en caché y los resultados por reunión se reanudan si la sesión se interrumpe.

**Persistencia (`PERSIST_DIR`):** resultados, trazas, caché del LLM y figuras se escriben en `PERSIST_DIR` (por defecto `/content/drive/MyDrive/d3_runs`; se monta Google Drive en la celda 1). Si Colab reinicia el entorno, basta volver a `Ejecutar todo`: las reuniones ya calculadas se reanudan desde Drive. Con `PERSIST_DIR = '/content/d3_runs'` todo queda en el disco efímero de la sesión.

In [ ]:
# 1. Configuración
MODE = 'DEMO'        # 'DEMO' | 'VALIDATION' | 'FULL'
LIMIT = 2            # solo VALIDATION: nº de reuniones (None = todas)
SOURCE = 'git'       # 'git' (clona REPO_URL; no se sube nada) | 'zip' (subir deliverable3_colab.zip)
REPO_URL = 'https://github.com/benjaminnalvear-dev/genai-vio-meeting-minutes'
COMMIT = ''          # solo SOURCE='git': hash o rama a hacer checkout ('' = HEAD)
MODEL = 'ministral-3:3b-instruct-2512-q4_K_M'
PERSIST_DIR = '/content/drive/MyDrive/d3_runs'  # resultados/trazas/caché sobreviven a un reinicio de Colab

ALL_CONDITIONS = 'baseline,d2,full,no_react,no_auditor,no_fewshot,no_cot,no_guards'
PLAN = {
    'DEMO':       dict(split='test',       conditions='full',    extra=['--demo']),
    'VALIDATION': dict(split='validation', conditions='full,d2', extra=(['--limit', str(LIMIT)] if LIMIT else [])),
    'FULL':       dict(split='test',       conditions=ALL_CONDITIONS, extra=[]),
}
assert MODE in PLAN, f'MODE invalido: {MODE}'
SPLIT, CONDITIONS, EXTRA = PLAN[MODE]['split'], PLAN[MODE]['conditions'], PLAN[MODE]['extra']

import os
if PERSIST_DIR.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
RESULTS = f'{PERSIST_DIR}/results/{SPLIT}'
TRACES = f'{PERSIST_DIR}/traces/{SPLIT}'
CACHE = f'{PERSIST_DIR}/cache/llm'
FIGURES = f'{PERSIST_DIR}/figures/{SPLIT}'
for d in (RESULTS, TRACES, CACHE, FIGURES):
    os.makedirs(d, exist_ok=True)
print('MODE =', MODE, '| split =', SPLIT, '| conditions =', CONDITIONS, '| extra =', EXTRA)
print('PERSIST_DIR =', PERSIST_DIR)

In [ ]:
# 2. GPU + Ollama + modelo (igual que la Entrega 2)
import subprocess, time, shutil, requests
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Esta sesión de Colab no tiene GPU. Ve a Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4 y vuelve a ejecutar.')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('Colab no detecta una GPU NVIDIA activa. Selecciona GPU T4 y reconecta.')
print('GPU detectada:', gpu.stdout.strip())
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
if not shutil.which('ollama'):
    subprocess.run(['bash', '-o', 'pipefail', '-c', 'curl -fsSL https://ollama.com/install.sh | sh'], check=True)
if not shutil.which('ollama'):
    raise RuntimeError('La instalación terminó sin dejar el ejecutable ollama en PATH')

ollama_log = open('/content/ollama_serve.log', 'a')
server = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        if requests.get('http://127.0.0.1:11434/api/tags', timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama no inició; revisa /content/ollama_serve.log')

print('Descargando modelo; solo se hace una vez por sesión de Colab...')
subprocess.run(['ollama', 'pull', MODEL], check=True)
subprocess.run(['ollama', '--version'], check=False)
print('Modelo:', MODEL)

In [ ]:
# 3. Obtener el código y dependencias
import os, subprocess, zipfile
from google.colab import files

if SOURCE == 'zip':
    up = files.upload()  # sube deliverable3_colab.zip
    zname = next(n for n in up if n.endswith('.zip'))
    if 'deliverable3' not in zname:
        raise ValueError(f'Subiste {zname}; en esta celda va deliverable3_colab.zip')
    with zipfile.ZipFile(zname) as z:
        z.extractall('/content')
elif SOURCE == 'git':
    subprocess.run(['rm', '-rf', '/content/repo'], check=True)
    subprocess.run(['git', 'clone', REPO_URL, '/content/repo'], check=True)
    if COMMIT:
        subprocess.run(['git', '-C', '/content/repo', 'checkout', COMMIT], check=True)
    subprocess.run(['rm', '-rf', '/content/deliverable3'], check=True)
    subprocess.run(['cp', '-r', '/content/repo/deliverable3', '/content/deliverable3'], check=True)
else:
    raise ValueError("SOURCE debe ser 'zip' o 'git'")

!pip install -q -r /content/deliverable3/requirements.txt
%cd /content/deliverable3
print(open('COMMIT.txt').read() if os.path.exists('COMMIT.txt') else 'git ' + COMMIT)

In [ ]:
# 4. Dataset (Fine Tuning.zip): con SOURCE='git' se usa el del repositorio; con 'zip' se sube
from pathlib import Path
REPO_DATA = Path('/content/repo/GenAI Constrained+Tool use/Fine Tuning.zip')
if SOURCE == 'git' and REPO_DATA.exists():
    DATA = str(REPO_DATA)
else:
    up = files.upload()
    dname = next(n for n in up if n.endswith('.zip'))
    if 'Fine' not in dname:
        raise ValueError(f'Subiste {dname}; en esta celda va Fine Tuning.zip')
    DATA = str(Path(dname).resolve())
print('DATA =', DATA)

In [ ]:
# 5. Ejecutar el experimento (salida en streaming; reanudable)
import json, subprocess, sys
from collections import deque
from pathlib import Path
cmd = [sys.executable, '-u', 'scripts/run_experiment.py', '--data', DATA, '--split', SPLIT,
       '--conditions', CONDITIONS, '--out', RESULTS, '--traces', TRACES, '--cache', CACHE] + EXTRA
print(' '.join(cmd), flush=True)
tail = deque(maxlen=40)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                        encoding='utf-8', errors='replace')
for line in proc.stdout:
    print(line, end='', flush=True)
    tail.append(line)
if proc.wait() != 0:
    raise RuntimeError(f'run_experiment.py terminó con código {proc.returncode}. Últimas líneas:\n' + ''.join(tail))

meta = json.loads(Path(f'{RESULTS}/summary.json').read_text(encoding='utf-8'))['meta']
if meta.get('failed'):
    print('\n' + '!' * 70)
    print(f"!!! {len(meta['failed'])} REUNIÓN(ES) FALLARON: vuelve a ejecutar esta celda para reintentarlas")
    for f in meta['failed']:
        print(f"!!!   [{f['condition']}] {f['meeting_id']}: {f['error']}")
    print('!' * 70)
else:
    print('\nSin reuniones fallidas.')
if meta.get('stale_results'):
    print('Resultados desactualizados (prompts/config cambiaron):', meta['stale_results'])

In [ ]:
# 6. Traza agente por agente (condición `full`)
import json
from pathlib import Path

def short(x, n=300):
    s = x if isinstance(x, str) else json.dumps(x, ensure_ascii=False)
    s = ' '.join(s.split())
    return s if len(s) <= n else s[:n] + ' ...'

trace_files = sorted(Path(f'{TRACES}/full').glob('*.jsonl'))
MID = 'M01' if Path(f'{TRACES}/full/M01.jsonl').exists() else (trace_files[0].stem if trace_files else None)
print('Reunión:', MID)
for line in Path(f'{TRACES}/full/{MID}.jsonl').read_text(encoding='utf-8').splitlines():
    ev = json.loads(line)
    agent = ev.get('agent')
    if agent == 'router':
        if ev.get('skipped'):
            print(f"\n=== [router] route={ev.get('route')}  revisión de '{ev['skipped']}' omitida ({ev.get('reason')})")
        else:
            aud = f"  (auditor_route={ev['auditor_route']})" if 'auditor_route' in ev else ''
            print(f"\n=== [router] route={ev.get('route')}{aud}  flagged={ev.get('flagged')}  next={ev.get('next')}")
        continue
    if agent == 'guard':
        print(f"=== [baranda {ev.get('guard')}] descarta {ev.get('problem')} sobre {ev.get('item_id')}: {ev.get('reason')}")
        continue
    if agent == 'finalize':
        print(f"\n=== [finalize] {ev.get('decisions')} decisiones, {ev.get('tasks')} tareas, "
              f"{ev.get('audit_rounds')} rondas de auditoría")
        print('  Acciones mecánicas:', ev.get('mechanical_actions') or 'ninguna')
        continue
    warn = '  [!] contexto cerca del límite' if ev.get('context_warning') else ''
    err = f"  [!] {ev['error']}" if ev.get('error') else ''
    print(f"\n[{agent} paso {ev.get('step')}] {ev.get('action')}   ({ev.get('seconds')}s, "
          f"{ev.get('prompt_tokens')} tokens de prompt, {ev.get('tokens')} generados){warn}{err}")
    if ev.get('thought'):
        print('  Pensamiento :', short(ev['thought']))
    for tc in ev.get('tool_calls') or []:
        print('  Herramienta :', short(tc, 200))
    for ob in ev.get('observations') or []:
        print('  Observación :', short(ob, 200))
    if ev.get('output') is not None:
        print('  Salida      :', short(ev['output'], 200))

In [ ]:
# 7. Acta final y métricas
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

acta = json.loads(Path(f'{RESULTS}/full/{MID}.json').read_text(encoding='utf-8'))
print(json.dumps(acta['parsed'], ensure_ascii=False, indent=2))

summary = json.loads(Path(f'{RESULTS}/summary.json').read_text(encoding='utf-8'))
conds = [c for c in summary if c not in ('meta', 'paired_full_vs_d2')]

def flat(d, pre=''):
    out = {}
    for k, v in d.items():
        if isinstance(v, dict):
            out.update(flat(v, f'{pre}{k}.'))
        elif isinstance(v, (int, float)) and not isinstance(v, bool):
            out[f'{pre}{k}'] = v
    return out

print('\nMétricas primarias')
display(pd.DataFrame({c: flat(summary[c]['primary']) for c in conds}).round(3))
print('Métricas extendidas')
display(pd.DataFrame({c: flat({k: v for k, v in summary[c]['extended'].items() if k != 'per_meeting'}) for c in conds}).round(3))

In [ ]:
# 8. Figuras
import subprocess, sys
from pathlib import Path
from IPython.display import Image, display
subprocess.run([sys.executable, 'scripts/make_figures.py', '--summary', f'{RESULTS}/summary.json', '--out', FIGURES], check=True)
for p in sorted(Path(FIGURES).glob('*.png')):
    print(p)
    display(Image(str(p)))

In [ ]:
# 9. Empaquetar y descargar results/, traces/ y figures/ (desde PERSIST_DIR; la caché no se incluye)
import subprocess
from pathlib import Path
from google.colab import files
folders = [d for d in ('results', 'traces', 'figures') if (Path(PERSIST_DIR) / d).exists()]
subprocess.run(['zip', '-qr', '/content/d3_outputs.zip'] + folders, check=True, cwd=PERSIST_DIR)
files.download('/content/d3_outputs.zip')